<a href="https://colab.research.google.com/github/AmplMrrr/compling-HW/blob/main/w2v_hw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

В этом практикуме мы рассмотрим работу с библиотекой **Gensim** для работы с векторными представлениями текста

Мы рассмотрим
- **Word2Vec** - векторные представления слов
- **FastText** - улучшенные представления с учетом морфологии  
- **Doc2Vec** - векторные представления документов


In [45]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

## Часть 1: Word2Vec

### Что такое Word2Vec?

Word2Vec преобразует слова в векторы чисел так, что семантически похожие слова оказываются близко в векторном пространстве.

**Два основных алгоритма:**
- **CBOW** - предсказывает слово по контексту
- **Skip-gram** - предсказывает контекст по слову

**Загрузка предобученной модели**

In [46]:
w2v_model = api.load('glove-wiki-gigaword-100')

print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

Размер словаря: 400000
Размерность векторов: 100


Найдите документацию `gensim`: какие датасеты кроме `glove-wiki-gigaword-100` доступны в библиотеке?

Выберите 3 датасета и кратко опишите их (источник данных, примерный объем, зачем такой датасет может использоваться)

Кроме датасета glove-wiki-gigaword-100 в библиотеки можно найти такие датасеты как:  fasttext-wiki-news-subwords-300, conceptnet-numberbatch-17-06-300, word2vec-ruscorpora-300, word2vec-google-news-300, glove-wiki-gigaword-50, glove-wiki-gigaword-200, glove-wiki-gigaword-300, glove-twitter-25, glove-twitter-50, glove-twitter-100, glove-twitter-200, __testing_word2vec-matrix-synopsis.

1. Описание датасета: word2vec-ruscorpora-300.
Источник: корпус русских текстов.
Примерные объем: несколько миллиардов слов.
Зачем: Материал для научного анализа семантики/грамматики/словоупотребления именно в русском языке. База для обучения, например, русскоязычных ботов.

2. Описание датасета: glove-twitter-100.
Источник: Twitter.
Примерный объем: 62 миллиона слов.
Зачем: Материал для научного анализа современного словоупотребления. Материал для sentiment analysis.

3. Описание датасета: word2vec-google-news-300.
Источник: основным источников является Google News Dataset.
Примерный объем: около 100 миллиардов слов.
Зачем: опять таки материал для научных исследований в сфере лингвистики (семантика, грамматика). Кроме того, датасет может использоваться как материал для анализа стиля написания новостных текстов.


**Базовые операции с векторами**

In [47]:
# Получаем вектор слова
vector = w2v_model['computer']
print(f"Вектор слова 'computer': {vector[:5]}...")  # Показываем первые 5 чисел

# Вычисляем схожесть между словами
similarity = w2v_model.similarity('computer', 'laptop')
print(f"Схожесть 'computer' и 'laptop': {similarity:.4f}")

Вектор слова 'computer': [-0.16298   0.30141   0.57978   0.066548  0.45835 ]...
Схожесть 'computer' и 'laptop': 0.7024


**Поиск похожих слов**

In [48]:
# Находим похожие слова
similar_words = w2v_model.most_similar('python', topn=5)
print("Слова, похожие на 'python':")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

Слова, похожие на 'python':
  monty: 0.6886
  php: 0.5865
  perl: 0.5784
  cleese: 0.5447
  flipper: 0.5113


*Ваш ответ здесь*

**Задание**

1. Загрузите любой датасет из gensim на ваш выбор

In [49]:
!pip install gensim

import gensim
import gensim.downloader as api
from gensim.models import Word2Vec, FastText, Doc2Vec
from gensim.models.doc2vec import TaggedDocument
import numpy as np

In [50]:
# загружаю датасет glove-twitter-25
w2v_model = api.load('glove-twitter-25')

# вывожу основную информацию о нем
print(f"Размер словаря: {len(w2v_model.key_to_index)}")
print(f"Размерность векторов: {w2v_model.vector_size}")

Размер словаря: 1193514
Размерность векторов: 25


2. Напишите функцию, которая принимает на вход любое слово и вовращает 10 наиболее близких по вектору слов

In [51]:
# вводится любое слово
word_u = input()

# функция ищет 10 слов, похожих на то, которое было введено, и выводит результат
similar_words = w2v_model.most_similar(word_u, topn=10)
print(f"Слова, похожие на {word_u}:")
for word, score in similar_words:
    print(f"  {word}: {score:.4f}")

woman
Слова, похожие на woman:
  child: 0.9372
  mother: 0.9215
  whose: 0.9175
  called: 0.9146
  person: 0.9136
  wife: 0.9088
  being: 0.9037
  father: 0.9028
  guy: 0.9026
  known: 0.8997


3. Обучите модель Word2Vec на тестовом датасете из ячейки ниже

Примените следующие настройки:

- размер вектора: 50
- размер окна: 3
- минимальная частота слова: 1
- потоков: 2
- использовать skip-gram

In [52]:
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [53]:
#задаю параметры модели
model = Word2Vec(
    sentences=cooking_sentences,
    vector_size=50,      # размерность векторов
    window=3,             # размер контекстного окна
    min_count=1,          # минимальная частота слова
    workers=2,            # количество ядер
    sg=1                  # 1 = Skip-Gram
)


In [54]:
print(f"Слова в словаре: {list(model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


4. Проверьте модель

In [55]:
# Проверяем похожие слова в кулинарной тематике
try:
    similar = model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

Слова, похожие на 'варить':
  вино: 0.2398
  ингредиенты: 0.2172
  хлеб: 0.1938
  брокколи: 0.1846
  кипятить: 0.1711


In [56]:
# Найдите слова, похожие на "духовка"
try:
    similar = model.wv.most_similar('духовка', topn=5)
    print("Слова, похожие на 'духовка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")

# Найдите слова, похожие на "овощи"
try:
    similar = model.wv.most_similar('овощи', topn=5)
    print("Слова, похожие на 'овощи':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")

Слова, похожие на 'духовка':
  ингредиенты: 0.3199
  десерт: 0.3064
  холодильник: 0.2705
  питание: 0.2243
  пирог: 0.2142
Слова, похожие на 'овощи':
  мариновать: 0.2716
  хлеб: 0.2691
  гриль: 0.2546
  фольга: 0.2409
  сахар: 0.2108


## Часть 2: FastText

FastText улучшает Word2Vec, рассматривая слова как наборы символов (n-грамм). Это позволяет работать с редкими словами и опечатками

5. Обучите FastText на корпусе текстов из пункта 3. Используйте код ниже

In [57]:
# дублирую корпус для удобства
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

In [58]:
# обучаю модель, меняю название переменной на название корпуса
ft_model = FastText(
    sentences=cooking_sentences,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2
)

In [59]:
print(f"Слова в словаре: {list(ft_model.wv.key_to_index.keys())[:10]}...")

Слова в словаре: ['овощи', 'мясо', 'соус', 'вода', 'тесто', 'духовка', 'специи', 'варить', 'брокколи', 'питание']...


6. Найдите слова, похожие на "варить", "духовка" и "овощи" с помощью обученной модели. Используйте код из пункта 4

In [60]:
# беру код из пункта 4 и выбираю новую модель ft_model
try:
    similar = ft_model.wv.most_similar('варить', topn=5)
    print("Слова, похожие на 'варить':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'варить' не найдено в словаре")

try:
    similar = ft_model.wv.most_similar('духовка', topn=5)
    print("Слова, похожие на 'духовка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'духовка' не найдено в словаре")


try:
    similar = ft_model.wv.most_similar('овощи', topn=5)
    print("Слова, похожие на 'овощи':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'овощи' не найдено в словаре")

Слова, похожие на 'варить':
  жарить: 0.5353
  парить: 0.4805
  месить: 0.3541
  тушить: 0.3405
  специи: 0.2622
Слова, похожие на 'духовка':
  взбивать: 0.4565
  лимон: 0.3561
  салат: 0.3050
  курица: 0.3041
  тост: 0.2944
Слова, похожие на 'овощи':
  жарить: 0.2960
  фольга: 0.2574
  морковь: 0.2297
  соус: 0.2172
  торт: 0.2094


7. Сравните модели

Дана функция для сравнения Word2Vec и FastText

Придумайте 3 слова с опечатками и проверьте, найдет ли их FastText и Word2Vec

In [61]:
def compare_models(word):
    """Сравнивает представления слова в разных моделях"""
    print(f"\nСравнение для слова: '{word}'")

    # Word2Vec
    try:
        w2v_similar = model.wv.most_similar(word, topn=2)
        print(f"  Word2Vec: {[w for w, _ in w2v_similar]}")
    except KeyError:
        print(f"  Word2Vec: слово не найдено")

    # FastText
    try:
        ft_similar = ft_model.wv.most_similar(word, topn=2)
        print(f"  FastText: {[w for w, _ in ft_similar]}")
    except KeyError:
        print(f"  FastText: слово не найдено")

# Сравниваем для разных слов
# пишу 4 слова для сравнения, два пишу неправильно, два с опечатками

compare_models('вено')
compare_models('мясить')
compare_models('броколи')
compare_models('чащка')

# справился только FastText и только со словом "брокколи"
# остальные неправильно написанные слова ни одна модель не нашла


Сравнение для слова: 'вено'
  Word2Vec: слово не найдено
  FastText: ['помидоры', 'молоко']

Сравнение для слова: 'мясить'
  Word2Vec: слово не найдено
  FastText: ['мясо', 'питание']

Сравнение для слова: 'броколи'
  Word2Vec: слово не найдено
  FastText: ['брокколи', 'молоко']

Сравнение для слова: 'чащка'
  Word2Vec: слово не найдено
  FastText: ['тушить', 'бекон']


## Часть 3: Doc2Vec

Doc2Vec расширяет Word2Vec для создания векторных представлений целых документов (предложений, абзацев, статей)

In [62]:
# Создаем размеченные документы
documents = [
    "machine learning is interesting",
    "deep learning uses neural networks",
    "python programming for data science",
    "artificial intelligence is amazing",
    "computer vision processes images"
]

# Преобразуем в формат TaggedDocument
tagged_docs = []
for i, doc in enumerate(documents):
    tokens = doc.split()
    tagged_doc = TaggedDocument(words=tokens, tags=[f"doc_{i}"])
    tagged_docs.append(tagged_doc)

print("Размеченные документы:")
for doc in tagged_docs[:3]:
    print(f"  Слова: {doc.words}")
    print(f"  Тег: {doc.tags}")

Размеченные документы:
  Слова: ['machine', 'learning', 'is', 'interesting']
  Тег: ['doc_0']
  Слова: ['deep', 'learning', 'uses', 'neural', 'networks']
  Тег: ['doc_1']
  Слова: ['python', 'programming', 'for', 'data', 'science']
  Тег: ['doc_2']


In [63]:
# Обучаем Doc2Vec
doc_model = Doc2Vec(
    documents=tagged_docs,
    vector_size=50,
    window=3,
    min_count=1,
    workers=2,
    epochs=20
)

print("Doc2Vec модель обучена!")
print(f"Количество документов: {len(doc_model.dv.key_to_index)}")

Doc2Vec модель обучена!
Количество документов: 5


In [64]:
# Получаем вектор документа
doc_vector = doc_model.dv["doc_0"]
print(f"Вектор документа doc_0: {doc_vector[:5]}...")

# Находим похожие документы
similar_docs = doc_model.dv.most_similar("doc_0", topn=2)
print("\nДокументы, похожие на doc_0:")
for doc_tag, similarity in similar_docs:
    doc_id = int(doc_tag.split('_')[1])
    print(f"  {doc_tag}: {similarity:.4f}")
    print(f"    Текст: {documents[doc_id]}")

Вектор документа doc_0: [-0.01057    -0.01198188 -0.01982618  0.01710627  0.00710373]...

Документы, похожие на doc_0:
  doc_1: 0.2735
    Текст: deep learning uses neural networks
  doc_2: 0.1275
    Текст: python programming for data science


In [65]:
# Сравниваем схожесть документов
def compare_documents(doc1_id, doc2_id):
    similarity = doc_model.dv.similarity(f"doc_{doc1_id}", f"doc_{doc2_id}")
    print(f"Схожесть doc_{doc1_id} и doc_{doc2_id}: {similarity:.4f}")
    print(f"  doc_{doc1_id}: {documents[doc1_id]}")
    print(f"  doc_{doc2_id}: {documents[doc2_id]}")

compare_documents(0, 1)  # machine learning vs deep learning
compare_documents(0, 3)  # machine learning vs AI

Схожесть doc_0 и doc_1: 0.2735
  doc_0: machine learning is interesting
  doc_1: deep learning uses neural networks
Схожесть doc_0 и doc_3: -0.0822
  doc_0: machine learning is interesting
  doc_3: artificial intelligence is amazing


8. Сравните схожесть doc_2 и doc_4

In [66]:
# немного дописываю код, чтобы сравнить документы 2 и 4
compare_documents(2, 4)

Схожесть doc_2 и doc_4: -0.0362
  doc_2: python programming for data science
  doc_4: computer vision processes images


9. Найдите самый похожий документ на doc_1

In [67]:
compare_documents(1, 0)
compare_documents(1, 2)
compare_documents(1, 3)
compare_documents(1, 4)

# самым похожим на doc_1 оказался doc_0
# показатель схожести = 0.2735

Схожесть doc_1 и doc_0: 0.2735
  doc_1: deep learning uses neural networks
  doc_0: machine learning is interesting
Схожесть doc_1 и doc_2: -0.0573
  doc_1: deep learning uses neural networks
  doc_2: python programming for data science
Схожесть doc_1 и doc_3: 0.2031
  doc_1: deep learning uses neural networks
  doc_3: artificial intelligence is amazing
Схожесть doc_1 и doc_4: -0.2546
  doc_1: deep learning uses neural networks
  doc_4: computer vision processes images


10. Выберите любую из трёх моделей. Обучите модели с разной размерностью (10, 50, 100). Продемонстрируйте качество их работы на примере поиска похожих слов (выберите любые 3 примера, соответствующих тематике корпуса из пункта 4)

In [68]:
# вновь копирую корпус для наглядности
cooking_sentences = [
    ['варить', 'суп', 'овощи', 'морковь', 'картофель'],
    ['жарить', 'курица', 'сковорода', 'масло', 'специи'],
    ['печь', 'хлеб', 'мука', 'дрожжи', 'духовка'],
    ['резать', 'овощи', 'салат', 'помидоры', 'огурцы'],
    ['смешивать', 'ингредиенты', 'тесто', 'яйца', 'молоко'],
    ['варить', 'паста', 'вода', 'соль', 'соус'],
    ['гриль', 'мясо', 'овощи', 'уголь', 'барбекю'],
    ['тушить', 'говядина', 'горшок', 'вино', 'травы'],
    ['запекать', 'рыба', 'лимон', 'духовка', 'фольга'],
    ['готовить', 'завтрак', 'яичница', 'бекон', 'тост'],
    ['месить', 'тесто', 'пирог', 'начинка', 'яблоки'],
    ['кипятить', 'вода', 'чай', 'кофе', 'чашка'],
    ['мариновать', 'мясо', 'соус', 'специи', 'холодильник'],
    ['взбивать', 'сливки', 'сахар', 'десерт', 'торт'],
    ['парить', 'овощи', 'здоровое', 'питание', 'брокколи']
]

# обучаю модель FastText
ft_model_10 = FastText(
    sentences=cooking_sentences,
    vector_size=10, # задаю размерность 10
    window=3,
    min_count=1,
    workers=2
)

print()
print("Размерность: 10")
print()

# выбираю три примера и тестирую модель
try:
    similar = ft_model_10.wv.most_similar('лосось', topn=5)
    print("Слова, похожие на 'лосось':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'лосось' не найдено в словаре")

try:
    similar = ft_model_10.wv.most_similar('морозильник', topn=5)
    print("Слова, похожие на 'морозильник':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'морозильник' не найдено в словаре")

try:
    similar = ft_model_10.wv.most_similar('индейка', topn=5)
    print("Слова, похожие на 'индейка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'индейка' не найдено в словаре")

# обучаю модель FastText
ft_model_50 = FastText(
    sentences=cooking_sentences,
    vector_size=50, # задаю размерность 50
    window=3,
    min_count=1,
    workers=2
)

print()
print("Размерность: 50")
print()

# выбираю три примера и тестирую модель
try:
    similar = ft_model_50.wv.most_similar('лосось', topn=5)
    print("Слова, похожие на 'лосось':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'лосось' не найдено в словаре")

try:
    similar = ft_model_50.wv.most_similar('морозильник', topn=5)
    print("Слова, похожие на 'морозильник':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'морозильник' не найдено в словаре")

try:
    similar = ft_model_50.wv.most_similar('индейка', topn=5)
    print("Слова, похожие на 'индейка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'индейка' не найдено в словаре")

# обучаю модель FastText
ft_model_100 = FastText(
    sentences=cooking_sentences,
    vector_size=100, # задаю размерность 100
    window=3,
    min_count=1,
    workers=2
)

print()
print("Размерность: 100")
print()

# выбираю три примера и тестирую модель
try:
    similar = ft_model_100.wv.most_similar('лосось', topn=5)
    print("Слова, похожие на 'лосось':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'лосось' не найдено в словаре")

try:
    similar = ft_model_100.wv.most_similar('морозильник', topn=5)
    print("Слова, похожие на 'морозильник':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'морозильник' не найдено в словаре")

try:
    similar = ft_model_100.wv.most_similar('индейка', topn=5)
    print("Слова, похожие на 'индейка':")
    for word, score in similar:
        print(f"  {word}: {score:.4f}")
except KeyError:
    print("Слово 'индейка' не найдено в словаре")

# прогон показал, что увеличение размерности точно хорошо работает в поиске слов, похожих своим написанием (морозильник/холодильник)
# однако, как кажется, увеличение размерности наоборот ухудшает результат, если речь идет о словах, похожих своими смыслами
# так, при размерности 10, слово "лосось" было похоже на слово "рыба", а слово "индейка" - на слово "курица", при увеличении размерности их связь уменьшилась или они вовсе пропали из списка похожих слов


Размерность: 10

Слова, похожие на 'лосось':
  холодильник: 0.7531
  говядина: 0.7262
  пирог: 0.6576
  рыба: 0.5725
  торт: 0.5365
Слова, похожие на 'морозильник':
  жарить: 0.7272
  масло: 0.6929
  мука: 0.6665
  травы: 0.5971
  варить: 0.5788
Слова, похожие на 'индейка':
  курица: 0.5626
  яичница: 0.5516
  варить: 0.5339
  завтрак: 0.5200
  говядина: 0.5068

Размерность: 50

Слова, похожие на 'лосось':
  тушить: 0.2415
  сковорода: 0.2241
  овощи: 0.1952
  хлеб: 0.1937
  печь: 0.1700
Слова, похожие на 'морозильник':
  холодильник: 0.4768
  ингредиенты: 0.3110
  масло: 0.2728
  яблоки: 0.2444
  специи: 0.2407
Слова, похожие на 'индейка':
  кофе: 0.2794
  курица: 0.2585
  гриль: 0.2066
  барбекю: 0.2013
  яйца: 0.1997

Размерность: 100

Слова, похожие на 'лосось':
  завтрак: 0.2008
  бекон: 0.1865
  кофе: 0.1718
  гриль: 0.1687
  фольга: 0.1675
Слова, похожие на 'морозильник':
  холодильник: 0.4327
  готовить: 0.2729
  тушить: 0.1782
  травы: 0.1657
  огурцы: 0.1458
Слова, похожие н